# Lab 04 — The Cached-Logit Pipeline: Off-Policy Distillation at Full Speed

**Tier 2 lab.** Part A executes and asserts anywhere; Part B trains only when
`RUN_TRAINING = True`.

**The question.** Lab 03 kept the teacher in memory for every training step, paying a teacher
forward pass per batch forever. But the teacher's distribution over a *fixed* corpus never
changes — so compute it once, store the top-k, and train the student against the cache with the
teacher evicted. This is the cheapest correct pipeline on bandwidth-limited hardware, because
its expensive phase is pure prefill (nobody decodes) and its training phase gives every byte of
memory to the student.

The price of the discount is a new class of silent failure. A live teacher cannot be misaligned
with the batch it just saw; a cache can — built from different token ids, a different
temperature, a different tokenizer version, or simply shuffled differently than the corpus in
hand. **A cache that cannot prove it belongs to your corpus is not an asset; it is a liability
with good storage characteristics.** So this lab is really two lessons in one: the pipeline,
and the paranoia. Part A builds and *attacks* the cache format on synthetic data — including
verifying that tampering is caught — before Part B ever touches a real teacher.

The pipeline:

```
stage 1 (pay once)   corpus ──> teacher prefill ──> top-k cache + manifest ──> disk
stage 2 (iterate)    cache + corpus ──> integrity checks ──> student training (no teacher)
```

In [1]:
import sys, os, json, math, shutil
sys.path.insert(0, "../code")

import numpy as np
import torch
import torch.nn.functional as F

from kd_core import (topk_forward_kl, make_topk_cache, kl_divergence,
                     shift_for_next_token, top1_agreement, mean_entropy,
                     bytes_per_token_cache, masked_mean)
from kd_pipeline import (set_seed_everywhere, config_fingerprint, MemoryPlan,
                         full_ft_gb, infer_gb, prefill_wallclock_hours,
                         TopKCacheWriter, TopKCacheReader, RunManifest)

RUN_TRAINING = False        # <-- flip on the training box
SEED = 17
set_seed_everywhere(SEED)
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"torch {torch.__version__} | device: {device} | RUN_TRAINING: {RUN_TRAINING}")

CFG = dict(
    teacher="HuggingFaceTB/SmolLM2-1.7B-Instruct",
    student="HuggingFaceTB/SmolLM2-360M-Instruct",
    k=64, cache_T=1.0, seq_len=384, vocab=49152,
    lr=3e-5, batch_size=8, grad_accum=4, max_steps=1500, warmup_steps=50,
    data="../data/lab03",           # Lab 03's corpus, reused for comparability
    cache_dir="../data/lab04_cache",
)
print(f"config fingerprint: {config_fingerprint({**CFG, 'seed': SEED})}")

torch 2.13.0+cpu | device: cpu | RUN_TRAINING: False
config fingerprint: b9e44266386f


## Part A · 1 — Price the cache before building it

Three numbers decide the design, and all three are computable before any model loads: bytes on
disk, prefill wall-clock, and the truncation bias at the chosen `k`. The first two are below;
the third was *measured* in Lab 02 §5 on this exact model family — k=64 kept >98% of the mass
with single-digit-percent KL bias in both estimator directions, which is why `CFG["k"] = 64`
and not a guess.

The memory plans for the two stages are the whole argument for this pipeline, so both are
asserted: stage 1 is teacher-heavy, stage 2 is teacher-*free*, and the student trains with more
than a hundred gigabytes of slack it did not have in Lab 03.

In [2]:
n_rows, T_len = 4096, CFG["seq_len"]
n_tokens = n_rows * T_len

s = bytes_per_token_cache(CFG["vocab"], k=CFG["k"])
cache_gb = s["topk_bytes_per_token"] * n_tokens / 1e9
dense_gb = s["dense_bytes_per_token"] * n_tokens / 1e9
prefill_min = prefill_wallclock_hours(n_tokens, prefill_tps=2000) * 60

print(f"corpus: {n_rows} rows x {T_len} tokens = {n_tokens/1e6:.2f} M positions")
print(f"dense cache would be {dense_gb:6.1f} GB  -> refused on principle")
print(f"top-{CFG['k']} cache        {cache_gb:6.2f} GB  ({s['compression']:.0f}x smaller)")
print(f"stage-1 prefill at ~2000 tok/s: ~{prefill_min:.0f} min, paid once")
assert cache_gb < 2.0, "this course-sized cache should be under 2 GB"

stage1 = (MemoryPlan(total_gb=128.0)
          .add("teacher 1.7B bf16 + activations", infer_gb(1.7) + 3.0)
          .add("cache write buffers", 2.0))
stage2 = (MemoryPlan(total_gb=128.0)
          .add("student 360M full fine-tune", full_ft_gb(0.36))
          .add("student activations", 4.0)
          .add("cache read (memory-mapped)", 0.5))
stage1.assert_fits(); stage2.assert_fits()
print(f"\nstage 1 plans {stage1.planned_gb:.1f} GB, stage 2 plans {stage2.planned_gb:.1f} GB "
      f"of 128 — the teacher's eviction is the pipeline's entire point")

corpus: 4096 rows x 384 tokens = 1.57 M positions
dense cache would be  154.6 GB  -> refused on principle
top-64 cache          0.61 GB  (255x smaller)
stage-1 prefill at ~2000 tok/s: ~13 min, paid once

stage 1 plans 8.4 GB, stage 2 plans 10.3 GB of 128 — the teacher's eviction is the pipeline's entire point


## Part A · 2 — Build the cache format, then attack it

`kd_pipeline.TopKCacheWriter/Reader` implement the on-disk layout. Before trusting them with an
hour of teacher prefill, this cell proves four properties on synthetic logits where the right
answers are computable:

1. **Round-trip fidelity.** Reading the cache back and computing `topk_forward_kl` matches
   computing it from the live logits via `make_topk_cache`, to fp16 storage precision.
2. **Fingerprint acceptance.** `verify_against` passes for the corpus the cache was built from.
3. **Tamper detection.** Change one token id in the corpus and `verify_against` must raise.
   A cache check that cannot fail is decoration, not verification.
4. **Storage honesty.** Bytes on disk match the arithmetic from A·1 within a few percent.

In [3]:
tmp = "../data/_lab04_selftest"
shutil.rmtree(tmp, ignore_errors=True)

B, T_len_t, V, k = 6, 40, 512, 16
g = torch.Generator().manual_seed(0)
t_logits = 6 * torch.randn(B, T_len_t, V, generator=g)
s_logits = 6 * torch.randn(B, T_len_t, V, generator=g)
ids = torch.randint(5, V, (B, T_len_t), generator=g)
m = torch.rand(B, T_len_t, generator=g) > 0.3

w = TopKCacheWriter(tmp, k=k, vocab_size=V, seq_len=T_len_t)
w.append(t_logits[:3], ids[:3], m[:3])
w.append(t_logits[3:], ids[3:], m[3:])
manifest = w.finalize()

r = TopKCacheReader(tmp)
batch = r.batch(range(B))

# 1: round-trip fidelity against the live computation.
live = make_topk_cache(t_logits, k=k)
kl_cache = topk_forward_kl(s_logits, batch, m, scale_by_T2=False)
kl_live  = topk_forward_kl(s_logits, live,  m, scale_by_T2=False)
assert abs(float(kl_cache) - float(kl_live)) < 2e-3, "fp16 round-trip must be faithful"

# 2 + 3: fingerprint accepts the true corpus, rejects a tampered one.
r.verify_against(ids.numpy())
tampered = ids.clone(); tampered[0, 0] += 1
try:
    r.verify_against(tampered.numpy())
    raise RuntimeError("tamper check FAILED to fail")
except AssertionError as e:
    print(f"tamper correctly rejected: {str(e).splitlines()[0][:70]}...")

# 4: storage honesty.
per_tok = bytes_per_token_cache(V, k=k)["topk_bytes_per_token"]
predicted = per_tok * B * T_len_t + 2 * B * T_len_t + B * T_len_t * (1 + 4)  # + tail fp16 dup, mask, ids
actual = manifest["bytes_on_disk"]
assert abs(actual - predicted) / predicted < 0.10, (actual, predicted)

print(f"round-trip KL {float(kl_cache):.4f} == live {float(kl_live):.4f}; "
      f"disk {actual} B ~= predicted {predicted} B")
print("cache format verified and attack-tested")

tamper correctly rejected: cache fingerprint 03ce833938caa3ad != corpus 3ca349aa115f6738: this ca...
round-trip KL 17.0463 == live 17.0464; disk 24720 B ~= predicted 25200 B
cache format verified and attack-tested


## Part A · 3 — The spot-check protocol

The fingerprint proves the cache matches the *token ids*. It cannot prove the cached
*log-probabilities* came from the teacher you think — a cache built at the wrong temperature,
from a stale checkpoint, or with a buggy shift passes the fingerprint check perfectly. The only
proof is re-derivation: sample rows, run the scorer again, compare.

The protocol is defined here as a function of an arbitrary `score_fn`, so it can be tested
*now* against a synthetic scorer with known behavior — both the accepting case and the
temperature-mismatch case Part B will guard against. In Part B the same function receives the
real teacher. Spot-check 1% of rows; the failure modes this catches are systematic, not
per-row, so a small sample catches them with near certainty.

In [4]:
def spot_check(reader, score_fn, rows, atol=1e-2):
    '''Re-derive cached top-k logprobs for `rows`; raise on mismatch.
    score_fn(input_ids [n,T]) -> logits [n,T,V].'''
    batch = reader.batch(rows)
    logits = score_fn(batch["input_ids"])
    log_p = F.log_softmax(logits.float() / reader.manifest["temperature"], dim=-1)
    recomputed = log_p.gather(-1, batch["topk_idx"])
    err = (recomputed - batch["topk_logprobs"]).abs()
    err = err[batch["mask"]].max() if batch["mask"].any() else err.max()
    assert float(err) < atol, f"spot check failed: max |Δlogprob| = {float(err):.4f}"
    return float(err)

honest_scorer = lambda x: t_logits[[int(i) for i in range(B)]][:len(x)]  # the true source
err = spot_check(r, lambda x: t_logits[:x.shape[0]], rows=range(3))
print(f"honest scorer accepted, max error {err:.5f} (fp16 storage noise)")

# The mismatch Part B must catch: same teacher, wrong temperature.
try:
    spot_check(r, lambda x: t_logits[:x.shape[0]] / 2.0, rows=range(3))
    raise RuntimeError("temperature mismatch NOT caught")
except AssertionError:
    print("temperature-mismatched scorer correctly rejected")
print("spot-check protocol verified — Part B reuses this exact function")

honest scorer accepted, max error 0.00390 (fp16 storage noise)
temperature-mismatched scorer correctly rejected
spot-check protocol verified — Part B reuses this exact function


## Part B — The two stages

Stage 1 is a loop of nothing but teacher prefill and `writer.append` — read it and notice what
is absent: no optimizer, no gradients, no student. Stage 2 is Lab 03's training loop with two
substitutions: the loss is `topk_forward_kl` against the cache batch, and there is no teacher
anywhere in the process. The `verify_against` + `spot_check` calls between the stages are not
ceremony; they are the pipeline's load-bearing wall.

A note on what the cache *cannot* do: it stores the teacher evaluated on **this corpus's
teacher-forced positions**, so it supports off-policy training only. The moment the student
generates its own tokens (Lab 07), the positions change and the cache is useless — that is not
a flaw, it is the definition of off-policy, and it is why this pipeline and on-policy
distillation are complements rather than competitors.

In [5]:
from transformers import AutoModelForCausalLM, get_cosine_schedule_with_warmup
import time

def stage1_build_cache(cfg):
    teacher = AutoModelForCausalLM.from_pretrained(
        cfg["teacher"], dtype=torch.bfloat16).to(device).eval()
    tr = torch.load(os.path.join(cfg["data"], "train.pt"))
    ids, mask = tr["input_ids"], tr["mask"]
    writer = TopKCacheWriter(cfg["cache_dir"], k=cfg["k"],
                             vocab_size=cfg["vocab"], seq_len=cfg["seq_len"],
                             temperature=cfg["cache_T"])
    bs, t0 = 16, time.time()
    with torch.no_grad():
        for i in range(0, len(ids), bs):
            logits = teacher(ids[i:i+bs].to(device)).logits
            writer.append(logits.cpu(), ids[i:i+bs], mask[i:i+bs])
    manifest = writer.finalize()
    del teacher
    if device == "cuda":
        torch.cuda.empty_cache()
    print(f"stage 1: {manifest['n_rows']} rows cached in {time.time()-t0:.0f}s, "
          f"{manifest['bytes_on_disk']/1e9:.2f} GB, fingerprint {manifest['corpus_fingerprint']}")
    return manifest

def stage2_train_from_cache(cfg):
    reader = TopKCacheReader(cfg["cache_dir"])
    tr = torch.load(os.path.join(cfg["data"], "train.pt"))
    reader.verify_against(tr["input_ids"].numpy())              # wall no. 1
    teacher_check = AutoModelForCausalLM.from_pretrained(       # wall no. 2
        cfg["teacher"], dtype=torch.bfloat16).to(device).eval()
    with torch.no_grad():
        spot_check(reader,
                   lambda x: teacher_check(x.to(device)).logits.cpu(),
                   rows=np.random.default_rng(0).choice(len(reader), 40, replace=False),
                   atol=5e-2)      # bf16 forward + fp16 storage; systematic bugs are >> this
    del teacher_check
    if device == "cuda":
        torch.cuda.empty_cache()
    print("integrity: fingerprint + spot check passed; teacher evicted")

    student = AutoModelForCausalLM.from_pretrained(
        cfg["student"], dtype=torch.bfloat16).to(device)
    opt = torch.optim.AdamW(student.parameters(), lr=cfg["lr"])
    sched = get_cosine_schedule_with_warmup(opt, cfg["warmup_steps"], cfg["max_steps"])
    order = np.random.default_rng(SEED).permutation(len(reader))
    log, step, t0 = [], 0, time.time()
    while step < cfg["max_steps"]:
        for i in range(0, len(order), cfg["batch_size"]):
            b = reader.batch(order[i:i+cfg["batch_size"]])
            ids_d = b["input_ids"].to(device)
            s_logits = student(ids_d).logits
            # shift: cache row t describes the distribution AT t (predicting t+1),
            # exactly like live logits — one shift for logits and mask together.
            cache_sh = {k2: (v[:, :-1].to(device) if v.dim() > 1 else v)
                        for k2, v in b.items() if k2 != "input_ids" and k2 != "mask"}
            loss = topk_forward_kl(s_logits[:, :-1], cache_sh,
                                   b["mask"][:, 1:].to(device)) / cfg["grad_accum"]
            loss.backward()
            if (step + 1) % cfg["grad_accum"] == 0:
                torch.nn.utils.clip_grad_norm_(student.parameters(), 1.0)
                opt.step(); sched.step(); opt.zero_grad()
            if step % 100 == 0:
                tps = (step + 1) * cfg["batch_size"] * cfg["seq_len"] / (time.time() - t0)
                log.append({"step": step, "loss": float(loss) * cfg["grad_accum"],
                            "train_tps": tps})
                print(f"step {step:>5}  cached-KL {log[-1]['loss']:.4f}  "
                      f"throughput {tps:,.0f} tok/s")
            step += 1
            if step >= cfg["max_steps"]:
                break
    fp = config_fingerprint({**cfg, "seed": SEED})
    out = f"../runs/lab04/cached_{fp}"
    os.makedirs(out, exist_ok=True)
    student.save_pretrained(out)
    json.dump(log, open(os.path.join(out, "log.json"), "w"), indent=2)
    RunManifest(name="cached-offpolicy", config=cfg, seed=SEED,
                artifacts_in={"cache": reader.manifest["corpus_fingerprint"]},
                artifacts_out={"checkpoint": out}).save(out)
    return out

if RUN_TRAINING:
    stage1_build_cache(CFG)
    out = stage2_train_from_cache(CFG)
    print("trained from cache ->", out)
else:
    print("RUN_TRAINING=False — Part B compiled but did not execute.")
    print("Stage 1 is minutes of prefill; stage 2 trains teacher-free thereafter.")

/usr/local/lib/python3.11/dist-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


RUN_TRAINING=False — Part B compiled but did not execute.
Stage 1 is minutes of prefill; stage 2 trains teacher-free thereafter.


## Part C — The verdict

**Expected ranges.**

- Final cached-KL in the same band as Lab 03's `mixed` arm forward-KL at equal steps (the
  objectives differ by the top-k truncation measured in Lab 02 §5 — single-digit percent at
  k=64, i.e. within run-to-run noise).
- Training throughput **1.5–3× Lab 03's**, with the exact factor depending on how large the
  teacher you evicted was relative to the student. The bigger the teacher, the bigger the win —
  which is precisely backwards from the live-teacher pipeline's economics, and the reason this
  is the default recipe for large-teacher off-policy work.
- Cache build: minutes, once. If you rebuild the cache more than once per corpus+teacher
  combination, something in your bookkeeping failed — the manifest exists so that never happens.

**Failure signatures.**

- *`verify_against` raises.* The corpus tensors changed since caching — re-tokenization, a
  different dataset shuffle, an edited seq_len. Rebuild the cache; do not "just comment out the
  check". The check firing is the system working.
- *Spot check fails with a uniform offset pattern.* Temperature mismatch between cache build
  and check — the exact scenario Part A·3 rehearsed. Fix `cache_T`, rebuild.
- *Spot check fails on scattered positions with large errors.* Wrong teacher revision or dtype
  drift. The manifest records what stage 1 used; diff it against what stage 2 loaded.
- *Cached-KL trains to a suspiciously perfect near-zero.* The student is likely being scored
  against its own top-k (a scorer/reader mix-up), or the mask includes almost nothing. Check
  the supervised-position count printed by the audit.
- *KL flat at a high value.* The off-by-one: cache row `t` describes the prediction of token
  `t+1`, same as live logits. If you shifted the cache but not the mask (or vice versa), Lab
  02 §3's invariant is broken — re-run that lab's assertion against a cached batch.

**Write the verdict.** Same discipline as Lab 03, plus one pipeline question: what is your
measured tokens/sec with and without the teacher resident, and does the ratio justify the cache
for *your* teacher size? For a 1.7B teacher the answer may be marginal; write down at what
teacher size it stops being marginal, because that number is now yours.

## Exercises

1. **Break the shift on purpose.** Train 200 steps with `b["mask"][:, :-1]` instead of
   `[:, 1:]` and watch what the loss curve does — and does not — tell you. This is the course's
   central bug, experienced deliberately in a controlled setting.
2. **k ablation, for real.** Rebuild the cache at k ∈ {8, 32, 128} and train 500 steps each.
   Plot final agreement vs cache size in GB. Where is *your* knee?
3. **Tail on, tail off.** `topk_forward_kl(use_tail=False)` renormalises instead of bucketing.
   Lab 02 measured the static bias; measure the *training* consequence — does the student that
   never sees the tail term behave differently on rare tokens? (Check entropy.)
4. **Cache at T=2.** Rebuild with `cache_T=2.0` and train; compare with T=1 caching + training.
   Where must the temperature agree, and where is it free? (The manifest stores it for a
   reason.)